In [ ]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 1000,
    "anticipatory_param": 0.1,
    "replay_interval": 4,  #
    "num_minibatches": 2,  # or 2, could be 2 minibatches per network, or 2 minibatches (1 for each network/player)
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    "residual_layers": [],
    "conv_layers": [],
    "dense_layer_widths": [128],
    "value_hidden_layer_widths": [],
    "advantage_hidden_layer_widths": [],
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    # "eg_epsilon_final": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    "sl_learning_rate": 0.005,
    "sl_momentum": 0.0,
    # "sl_weight_decay": 1e-9,
    # "sl_clipnorm": 1.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    "sl_residual_layers": [],
    "sl_conv_layers": [],
    "sl_dense_layer_widths": [128],
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}
config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp"))
trainer.checkpoint_interval = 100
trainer.test_interval = 100
trainer.test_trials = 100
trainer.train()

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


NFSPDQNConfig
Using default save_intermediate_weights     : False
Using         training_steps                : 1000
Using default adam_epsilon                  : 1e-08
Using         momentum                      : 0.0
Using         learning_rate                 : 0.1
Using         clipnorm                      : 10.0
Using         optimizer                     : <class 'torch.optim.sgd.SGD'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 2
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 128
Using         replay_buffer_size            : 1000
Using         min_replay_buffer_size        : 500
Using         n_step                        : 1
Using default discount_factor               : 0.99
Using         per_alpha                     : 0.0
Using    

KeyboardInterrupt: 

In [ ]:
# shared network but not shared buffer?
# 1 vs 2 minibatches

from nfsp_agent_clean import NFSPDQN
from agent_configs import NFSPDQNConfig
from game_configs import LeducHoldemConfig, MatchingPenniesConfig
from utils import KLDivergenceLoss, CategoricalCrossentropyLoss, HuberLoss, MSELoss
from torch.optim import Adam, SGD

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 50000,
    "anticipatory_param": 0.1,
    "replay_interval": 128,  #
    "num_minibatches": 1,  # or 2, could be 2 minibatches per network, or 2 minibatches (1 for each network/player)
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": MSELoss(),
    "min_replay_buffer_size": 1000,
    "minibatch_size": 128,
    "replay_buffer_size": 2e5,
    "transfer_interval": 300,
    "residual_layers": [],
    "conv_layers": [],
    "dense_layer_widths": [128],
    "value_hidden_layer_widths": [],
    "advantage_hidden_layer_widths": [],
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    # "eg_epsilon_final": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    "sl_learning_rate": 0.005,
    "sl_momentum": 0.0,
    # "sl_weight_decay": 1e-9,
    # "sl_clipnorm": 1.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 1000,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 2000000,
    "sl_residual_layers": [],
    "sl_conv_layers": [],
    "sl_dense_layer_widths": [128],
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}
config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=LeducHoldemConfig(),
)
config.save_intermediate_weights = False

In [ ]:
from pettingzoo.classic import leduc_holdem_v4
from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)


env = leduc_holdem_v4.env()
# env = matching_pennies_env(render_mode="human", max_cycles=1)

print(env.observation_space("player_0"))

agent = NFSPDQN(env, config, name="NFSP-LeducHoldem-Standard", device="cpu")

In [ ]:
agent.checkpoint_interval = 2000
agent.checkpoint_trials = 10000
agent.train()